In [40]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from scipy.stats import norm
from scipy.linalg import expm
import yfinance as yf


#Black Scholes Option Pricing Formula 
def black_scholes_price(S0, K, T, r, sigma, option_type='call'):
    """
    Standard Black-Scholes option pricing formula.
    """
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == ['call', 'Call']:
        price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
        
    return price

# Getting the data for Apple
ticker_symbol = "AAPL"
stock = yf.Ticker(ticker_symbol)
data = stock.history(period="5y")
df = data.copy()
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

# Defining a Hidden Markov Model for Calculating the Volatility 

#--------------------------------------------------------------------
# We define two states of Volatility, Low Volatility (Bull Market) 
# and High Volatility (Bear Market), indicated by state 0 and 1
# respectively. 
#--------------------------------------------------------------------

def calculate_hmm_volatility(df, price_col='Close', T=0.5, n_regimes=2):
    prices = df[price_col].values
    log_returns = np.diff(np.log(prices)).reshape(-1, 1)
    
    # 1. Fit Gaussian HMM, Assuming Normal Distribution for each state
    hmm = GaussianHMM(n_components=n_regimes, covariance_type="diag", n_iter=1000, random_state=42)
    hmm.fit(log_returns)
    
    # Extract Daily Volatilities per regime
    daily_stds = np.sqrt(hmm.covars_.flatten())
    
    # Sort regimes by volatility: State 0 = Low Vol, State 1 = High Vol
    sort_idx = np.argsort(daily_stds)
    daily_stds = daily_stds[sort_idx]
    P_daily = hmm.transmat_[sort_idx][:, sort_idx]
    
    # Annualized volatilities per regime
    trading_days = 252
    sigma_regimes = daily_stds * np.sqrt(trading_days)
    
    # 2. Estimate initial state distribution (posterior probability of current day)
    hidden_states = hmm.predict(log_returns)
    state_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sort_idx)}
    current_regime = state_map[hidden_states[-1]]
    
    # Initial state probability vector
    pi_0 = np.zeros(n_regimes)
    pi_0[current_regime] = 1.0
    
    # 3. Project average time spent in each regime over maturity T (n_steps)
    n_steps = int(T * trading_days)
    
    # Sum of transition probabilities over n_steps
    # Compute expected time proportion in each regime
    state_occupancy = np.zeros(n_regimes)
    current_p = np.eye(n_regimes)
    
    for _ in range(n_steps):
        state_occupancy += pi_0 @ current_p
        current_p = current_p @ P_daily
        
    regime_weights = state_occupancy / n_steps
    
    # 4. Compute expected forward variance & volatility
    # Var(r) = w_0 * sigma_0^2 + w_1 * sigma_1^2
    expected_variance = np.sum(regime_weights * (sigma_regimes**2))
    hmm_effective_vol = np.sqrt(expected_variance)
    
    print("=== HMM VOLATILITY ESTIMATION ===")
    print(f"Dataset End Date: {df.index[-1].strftime('%Y-%m-%d')}")
    print(f"Current Spot Price (S0): ${prices[-1]:.2f}")
    print(f"Active Regime: State {current_regime} ({'Low Vol' if current_regime == 0 else 'High Vol'})")
    print(f"Regime Annualized Volatilities: Low Vol = {sigma_regimes[0]:.2%}, High Vol = {sigma_regimes[1]:.2%}")
    print(f"Expected Time Allocation over T={T} years: State 0: {regime_weights[0]:.1%}, State 1: {regime_weights[1]:.1%}")
    print(f"HMM Effective Volatility (sigma_HMM): {hmm_effective_vol:.2%}\n")
    
    return prices[-1], hmm_effective_vol, sigma_regimes



if __name__ == "__main__":
    n_days = 756
    # Historical dataset ending 1 month ago (July 2026)
    dates = pd.date_range(end='2026-07-24', start = '2023-07-24', periods=n_days)
    # Parameters
    T = 0.5          # 6 Months to maturity
    r = 0.04         # 4% Risk-free rate
    strikes = [300, 305, 310, 315, 320]
    
    # 1. Calculate HMM-weighted Volatility
    S0, sigma_hmm, sigma_regimes = calculate_hmm_volatility(df, price_col='Close', T=T)
    
    # 2. Calculate Standard Historical Volatility (Simple 30-day trailing)
    daily_returns = np.diff(np.log(df['Close'].values))
    sigma_hist = np.std(daily_returns[-30:]) * np.sqrt(252)
    
    # 3. Price European Call Options across strikes
    results = []
    for K in strikes:
        price_hmm = black_scholes_price(S0, K, T, r, sigma=sigma_hmm, option_type='call')
        price_hist = black_scholes_price(S0, K, T, r, sigma=sigma_hist, option_type='call')
        
        results.append({
            'Strike ($)': K,
            'BS Price (HMM Vol)': round(price_hmm, 2),
            'BS Price (30d Hist Vol)': round(price_hist, 2),
            'Price Difference ($)': round(price_hmm - price_hist, 2)
        })
        
    results_df = pd.DataFrame(results)
    
    print("=== BLACK-SCHOLES OPTION PRICES COMPARISON ===")
    print(f"HMM Volatility Input:       {sigma_hmm:.2%}")
    print(f"30-Day Historical Vol Input: {sigma_hist:.2%}\n")
    print(results_df.to_string(index=False))

=== HMM VOLATILITY ESTIMATION ===
Dataset End Date: 2026-08-24
Current Spot Price (S0): $310.34
Active Regime: State 0 (Low Vol)
Regime Annualized Volatilities: Low Vol = 21.64%, High Vol = 54.62%
Expected Time Allocation over T=0.5 years: State 0: 86.4%, State 1: 13.6%
HMM Effective Volatility (sigma_HMM): 28.45%

=== BLACK-SCHOLES OPTION PRICES COMPARISON ===
HMM Volatility Input:       28.45%
30-Day Historical Vol Input: 32.29%

 Strike ($)  BS Price (HMM Vol)  BS Price (30d Hist Vol)  Price Difference ($)
        300               16.94                    20.08                 -3.15
        305               19.14                    22.37                 -3.23
        310               21.50                    24.79                 -3.29
        315               24.02                    27.35                 -3.33
        320               26.70                    30.05                 -3.35
